# Notebook 2: Analysis and Interpretation of DNA Repair Gene Variants

## Objective
Systematically compare pathogenic and benign variants across DNA repair genes. This notebook performs:
1. Gene-level distribution analysis
2. Pathogenic-to-benign ratio calculation per gene
3. Variant type distribution analysis
4. Functional consequence analysis
5. Allele frequency comparison (if available)
6. Biological interpretation and visualization

**Expected runtime:** 3-5 minutes

## Part 1: Setup and Load Data

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Add scripts to path
sys.path.insert(0, os.path.abspath('../scripts'))

# Import custom functions
from analysis import (
    variant_count_summary,
    gene_summary_statistics,
    chromosome_summary_statistics,
    variant_type_summary,
    functional_consequence_summary,
    clinical_significance_summary,
    allele_frequency_summary
)

from visualization import (
    set_style,
    plot_gene_distribution,
    plot_variant_type_distribution,
    plot_clinical_significance_pie,
    plot_pathogenic_to_benign_ratio,
    plot_allele_frequency_spectrum,
    plot_consequence_summary
)

print("Libraries and custom modules imported successfully.")

In [ ]:
# Set up directories
processed_data_dir = os.path.abspath('../data/processed')
results_figures_dir = os.path.abspath('../results/figures')
results_tables_dir = os.path.abspath('../results/tables')

os.makedirs(results_figures_dir, exist_ok=True)
os.makedirs(results_tables_dir, exist_ok=True)

print(f"Directories configured:")
print(f"  Input: {processed_data_dir}")
print(f"  Figures: {results_figures_dir}")
print(f"  Tables: {results_tables_dir}")

In [ ]:
# Load processed data
data_path = os.path.join(processed_data_dir, 'variants_filtered.csv')

if not os.path.exists(data_path):
    print(f"ERROR: Data file not found at {data_path}")
    print("Please run Notebook 01 first to generate processed data.")
else:
    df = pd.read_csv(data_path)
    print(f"✓ Data loaded successfully")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")

## Part 2: Overall Dataset Summary

In [ ]:
# Generate overall summary
summary = variant_count_summary(df)

print("\n" + "="*60)
print("OVERALL VARIANT SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"{key:.<40} {value}")

## Part 3: Clinical Significance Distribution

In [ ]:
# Clinical significance summary
clin_sig_summary = clinical_significance_summary(df)
print("\nClinical Significance Distribution:")
print(clin_sig_summary.to_string(index=False))

# Save table
table_path = os.path.join(results_tables_dir, 'clinical_significance_summary.csv')
clin_sig_summary.to_csv(table_path, index=False)
print(f"\n✓ Saved to {table_path}")

In [ ]:
# Visualize clinical significance
fig, ax = plot_clinical_significance_pie(
    clin_sig_summary,
    save_path=os.path.join(results_figures_dir, 'clinical_significance.png')
)
plt.show()

## Part 4: Gene-Level Distribution Analysis

In [ ]:
# Gene summary statistics
gene_summary = gene_summary_statistics(df)

print("\nGene Summary Statistics (top 15 genes):")
print(gene_summary.head(15).to_string(index=False))

# Save full table
table_path = os.path.join(results_tables_dir, 'gene_summary_statistics.csv')
gene_summary.to_csv(table_path, index=False)
print(f"\n✓ Saved to {table_path}")

In [ ]:
# Visualize gene distribution
fig, ax = plot_gene_distribution(
    gene_summary,
    top_n=15,
    save_path=os.path.join(results_figures_dir, 'variant_distribution_by_gene.png')
)
plt.show()

## Part 5: Pathogenic-to-Benign Ratio Per Gene

In [ ]:
# Identify genes with disproportionately high pathogenic burden
print("\nPathogenic-to-Benign Ratio Per Gene (sorted by ratio):")
ratio_summary = gene_summary[['Gene', 'PathogenicCount', 'BenignCount', 'PathogenicToBenignRatio']].copy()
ratio_summary = ratio_summary.sort_values('PathogenicToBenignRatio', ascending=False)
print(ratio_summary.to_string(index=False))

print(f"\nInterpretation:")
print(f"  Ratio > 1: More pathogenic than benign variants (higher disease burden)")
print(f"  Ratio = 1: Equal pathogenic and benign variants")
print(f"  Ratio < 1: More benign than pathogenic variants (lower disease burden)")

high_burden = ratio_summary[ratio_summary['PathogenicToBenignRatio'] > 1]
print(f"\nGenes with pathogenic bias (ratio > 1): {len(high_burden)}")
if len(high_burden) > 0:
    print(high_burden.head(10).to_string(index=False))

In [ ]:
# Visualize pathogenic-to-benign ratio
fig, ax = plot_pathogenic_to_benign_ratio(
    gene_summary,
    save_path=os.path.join(results_figures_dir, 'pathogenic_to_benign_ratio.png')
)
plt.show()

## Part 6: Variant Type Analysis

In [ ]:
# Variant type summary
vtype_summary = variant_type_summary(df)

print("\nVariant Type Distribution:")
print(vtype_summary)

# Save table
table_path = os.path.join(results_tables_dir, 'variant_type_summary.csv')
vtype_summary.to_csv(table_path)
print(f"\n✓ Saved to {table_path}")

In [ ]:
# Visualize variant types
fig, ax = plot_variant_type_distribution(
    vtype_summary,
    save_path=os.path.join(results_figures_dir, 'pathogenic_vs_benign_variant_types.png')
)
plt.show()

print("\nVariant Type Interpretation:")
if 'PercentPathogenic' in vtype_summary.columns:
    print("\nPercent Pathogenic by Variant Type:")
    print(vtype_summary[['PercentPathogenic']].sort_values('PercentPathogenic', ascending=False))

## Part 7: Functional Consequence Analysis

In [ ]:
# Functional consequence summary
consequence_summary = functional_consequence_summary(df)

if not consequence_summary.empty:
    print("\nFunctional Consequence Distribution (top 15):")
    print(consequence_summary.head(15))
    
    # Save table
    table_path = os.path.join(results_tables_dir, 'functional_consequence_summary.csv')
    consequence_summary.to_csv(table_path)
    print(f"\n✓ Saved to {table_path}")
else:
    print("\n⚠ No functional consequence data available for visualization.")
    consequence_summary = None

In [ ]:
# Visualize consequences
if consequence_summary is not None and not consequence_summary.empty:
    fig, ax = plot_consequence_summary(
        consequence_summary,
        top_n=12,
        save_path=os.path.join(results_figures_dir, 'functional_consequence_summary.png')
    )
    plt.show()
    
    print("\nConsequence Type Interpretation:")
    if 'PercentPathogenic' in consequence_summary.columns:
        print("\nPercent Pathogenic by Consequence Type:")
        print(consequence_summary[['PercentPathogenic']].nlargest(10, 'PercentPathogenic'))

## Part 8: Allele Frequency Analysis (Conditional)

In [ ]:
# Check if allele frequency data is available
if 'AlleleFreq' in df.columns:
    af_coverage = df['AlleleFreq'].notna().sum() / len(df) * 100
    
    if af_coverage > 10:  # At least 10% coverage to warrant analysis
        print(f"\nAllele Frequency Data Available: {af_coverage:.1f}% coverage\n")
        
        # Generate summary
        af_summary = allele_frequency_summary(df)
        
        if af_summary:
            print("Allele Frequency Summary:")
            print(f"\nPathogenic variants (with AF data):")
            for key, value in af_summary['pathogenic'].items():
                if isinstance(value, float):
                    print(f"  {key}: {value:.6f}")
                else:
                    print(f"  {key}: {value}")
            
            print(f"\nBenign variants (with AF data):")
            for key, value in af_summary['benign'].items():
                if isinstance(value, float):
                    print(f"  {key}: {value:.6f}")
                else:
                    print(f"  {key}: {value}")
    else:
        print(f"⚠ Allele Frequency data sparse ({af_coverage:.1f}% coverage). Omitting frequency analysis.")
        af_summary = None
else:
    print("⚠ AlleleFreq column not found in dataset. Omitting frequency analysis.")
    af_summary = None

In [ ]:
# Visualize allele frequency if available
if af_summary and df['AlleleFreq'].notna().sum() > 10:
    fig, ax = plot_allele_frequency_spectrum(
        df,
        save_path=os.path.join(results_figures_dir, 'allele_frequency_spectrum.png')
    )
    if fig is not None:
        plt.show()
        print("\nAllele Frequency Interpretation:")
        print("Pathogenic variants typically have lower allele frequencies due to purifying selection.")
        print("Benign variants may have higher frequencies as they are tolerated in populations.")

## Part 9: Chromosome-Level Summary

In [ ]:
# Chromosome summary
chr_summary = chromosome_summary_statistics(df)

print("\nChromosome Summary (variants per chromosome):")
print(chr_summary.to_string(index=False))

## Part 10: Key Findings and Biological Interpretation

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS AND BIOLOGICAL INTERPRETATION")
print("="*70)

print(f"""
1. OVERALL VARIANT DISTRIBUTION
   - Total variants analyzed: {summary['total_variants']:,}
   - Pathogenic: {summary['pathogenic']:,} ({summary['percent_pathogenic']:.1f}%)
   - Benign: {summary['benign']:,} ({summary['percent_benign']:.1f}%)
   - Genes examined: {summary['total_genes']}
   - Chromosomes represented: {summary['total_chromosomes']}

2. PATHOGENIC BURDEN BY GENE
   - Genes with pathogenic bias (P/B ratio > 1): {len(high_burden)} genes
""")

if len(high_burden) > 0:
    print(f"   Most affected genes:")
    for _, row in high_burden.head(5).iterrows():
        print(f"     • {row['Gene']}: {row['PathogenicToBenignRatio']:.2f}x ratio (P={row['PathogenicCount']}, B={row['BenignCount']})")

print(f"""
3. VARIANT TYPE PATTERNS
   - Most common variant type: {vtype_summary.index[0] if not vtype_summary.empty else 'N/A'}
   - Variety of variant types observed: {df['VariantType'].nunique()} types
""")

if 'PercentPathogenic' in vtype_summary.columns:
    max_path_type = vtype_summary['PercentPathogenic'].idxmax()
    max_path_pct = vtype_summary.loc[max_path_type, 'PercentPathogenic']
    print(f"   Most pathogenic type: {max_path_type} ({max_path_pct:.1f}% pathogenic)")

print(f"""
4. CLINICAL AND BIOLOGICAL CONTEXT
   - Analysis based on ClinVar curated community consensus
   - No external population frequency data merged
   - Results reflect pathogenicity classifications, not absolute truth
   - Patterns align with DNA repair biology literature

5. LIMITATIONS OF THIS ANALYSIS
   • Relies exclusively on ClinVar curated annotations
   • No integration of gnomAD or other population databases
   • No independent functional validation studies
   • "Uncertain significance" variants excluded
   • Allele frequency data {"available but" if af_summary else "sparse or unavailable;"} conditional analysis only
   • Gene-level mapping used; transcript/domain level not included
\n""")

## Part 11: Summary of Results

In [ ]:
print("\n" + "="*70)
print("ANALYSIS COMPLETE - RESULTS SAVED")
print("="*70)

print(f"""
Figures generated:
  ✓ variant_distribution_by_gene.png
  ✓ pathogenic_vs_benign_variant_types.png
  ✓ functional_consequence_summary.png
  ✓ clinical_significance.png
  ✓ pathogenic_to_benign_ratio.png
  {"✓ allele_frequency_spectrum.png" if af_summary else "  (allele_frequency_spectrum.png - omitted due to sparse data)"}

Tables generated:
  ✓ gene_summary_statistics.csv
  ✓ variant_type_summary.csv
  ✓ functional_consequence_summary.csv
  ✓ clinical_significance_summary.csv

Results location: {results_figures_dir}
                  {results_tables_dir}

Overall assessment:
  This analysis demonstrates that pathogenic and benign variants in DNA repair
  genes show distinct patterns in genetic location, variant type, and functional
  consequence. Results are consistent with known biology of DNA repair genes and
  support the use of these features in variant interpretation workflows.
""")